In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [24]:
data_dir = '/content/drive/MyDrive/AI_model/src/processed_data'

In [25]:
X_train = joblib.load(
    f'{data_dir}/X_train.pkl'
)

X_val = joblib.load(
    f'{data_dir}/X_val.pkl'
)

y_train = joblib.load(
    f'{data_dir}/y_train.pkl'
)

y_val = joblib.load(
    f'{data_dir}/y_val.pkl'
)

In [26]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)


print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (1600, 20)
X_val: (400, 20)
y_train: (1600,)
y_val: (400,)


In [27]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [28]:
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

In [29]:
models = {
    'Logistic Regression': logistic_model,
    'KNN': knn_model
}

In [30]:
print(models)

{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42), 'KNN': KNeighborsClassifier()}


In [31]:
results = []

for name, model in models.items():

    print(f"Đang train: {name}")
    print("y_train dtype:", y_train.dtype)
    print("y_train unique:", sorted(y_train.unique()))
    print("y_train min:", y_train.min())
    print("y_train max:", y_train.max())

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    precision = precision_score(
        y_val,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_val,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_val,
        y_pred,
        average='weighted'
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1
    })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("-" * 50)

Đang train: Logistic Regression
y_train dtype: int64
y_train unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
y_train min: 0
y_train max: 3
Accuracy: 0.9750
Precision: 0.9759
Recall: 0.9750
F1-score: 0.9750
--------------------------------------------------
Đang train: KNN
y_train dtype: int64
y_train unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
y_train min: 0
y_train max: 3
Accuracy: 0.5300
Precision: 0.5698
Recall: 0.5300
F1-score: 0.5407
--------------------------------------------------


In [32]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.975,0.975946,0.975,0.975020
1,KNN,0.530,0.569762,0.530,0.540707


In [34]:
model_train = "";
for model_train in models:
  print(model_train)
  rf_model = models[model_train]

  y_pred_rf = rf_model.predict(X_val)

  print(
      classification_report(
          y_val,
          y_pred_rf
      )
  )

Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       105
           1       0.94      1.00      0.97        91
           2       0.99      0.95      0.97        92
           3       0.97      0.99      0.98       112

    accuracy                           0.97       400
   macro avg       0.98      0.97      0.97       400
weighted avg       0.98      0.97      0.98       400

KNN
              precision    recall  f1-score   support

           0       0.70      0.72      0.71       105
           1       0.34      0.45      0.39        91
           2       0.35      0.37      0.36        92
           3       0.81      0.54      0.65       112

    accuracy                           0.53       400
   macro avg       0.55      0.52      0.53       400
weighted avg       0.57      0.53      0.54       400



In [35]:
results_path = f'{data_dir}/model_results.csv'

results_df.to_csv(
    results_path,
    index=False
)

print("Đã lưu:", results_path)

Đã lưu: /content/drive/MyDrive/AI_model/src/processed_data/model_results.csv


In [36]:
model_dir = '/content/drive/MyDrive/AI_model/model'

os.makedirs(
    model_dir,
    exist_ok=True
)

In [37]:
for name, model in models.items():

    filename = name.lower().replace(
        ' ',
        '_'
    )

    joblib.dump(
        model,
        f'{model_dir}/{filename}.pkl'
    )

print("Đã lưu tất cả model!")

Đã lưu tất cả model!


In [38]:
best_model_name = results_df.loc[
    results_df['F1-score'].idxmax(),
    'Model'
]

best_model = models[best_model_name]

print("Model có F1-score cao nhất:")
print(best_model_name)

Model có F1-score cao nhất:
Logistic Regression


In [39]:
joblib.dump(
    best_model,
    f'{model_dir}/best_model.pkl'
)

['/content/drive/MyDrive/AI_model/model/best_model.pkl']